# Partie 2 -- Entrainement des Modeles de Boosting

## Prudential Life Insurance Assessment

**Objectif** : Entrainer XGBoost et CatBoost pour predire le niveau de risque (`Response`, 1 a 8).

---

### 4 configurations testees

| # | Modele | Methode |
|:--|:-------|:--------|
| 1 | XGBoost | Arrondi simple |
| 2 | XGBoost | Avec offset optimization |
| 3 | CatBoost | Arrondi simple |
| 4 | CatBoost | Avec offset optimization |

In [1]:
!pip install pandas numpy scikit-learn xgboost catboost scipy


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor
from scipy.optimize import minimize_scalar
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
import warnings
import os

warnings.filterwarnings("ignore")
NUM_CLASSES = 8

---
## Fonctions utilitaires

In [3]:
def qwk(y_pred, y_true):
    """Quadratic Weighted Kappa."""
    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred)
    y_pred = np.clip(np.round(y_pred), np.min(y_true), np.max(y_true)).astype(int)
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


def optimize_offsets(preds, labels):
    """
    Optimise un offset par classe pour maximiser le QWK.
    Utilise minimize_scalar (un seul parametre a la fois).
    """
    preds = np.array(preds, dtype=float)
    labels = np.array(labels, dtype=float)
    adjusted = preds.copy()
    offsets = np.zeros(NUM_CLASSES)
    
    for j in [6, 4, 5, 3, 2, 1, 7, 0]:
        def objective(x, cls=j):
            adjusted[preds.astype(int) == cls] = preds[preds.astype(int) == cls] + x
            return -qwk(adjusted, labels)
        
        result = minimize_scalar(objective, bounds=(-3, 3), method="bounded")
        offsets[j] = result.x
        adjusted[preds.astype(int) == j] = preds[preds.astype(int) == j] + offsets[j]
    
    return offsets


def apply_offsets(preds, offsets):
    """Applique les offsets et retourne les predictions finales (1-8)."""
    preds = np.array(preds, dtype=float)
    adjusted = preds.copy()
    for j in range(NUM_CLASSES):
        mask = preds.astype(int) == j
        adjusted[mask] = preds[mask] + offsets[j]
    return np.clip(np.round(adjusted), 1, 8).astype(int)


print("Fonctions definies.")

Fonctions definies.


---
## 1. Chargement des donnees

In [4]:
DATA_DIR = "prudential-life-insurance-assessment"
TARGET = "Response"

train = pd.read_csv(os.path.join(DATA_DIR, "train_clean.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test_clean.csv"))

feature_cols = [c for c in train.columns if c not in ["Id", TARGET]]
X = train[feature_cols].values
y = train[TARGET].values
X_test = test[feature_cols].values
test_ids = test["Id"].values

print(f"Train : {X.shape}")
print(f"Test  : {X_test.shape}")
print(f"Features : {len(feature_cols)}")

Train : (59381, 127)
Test  : (19765, 127)
Features : 127


---
## 2. Split train / validation (80/20)

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")

Train : (47504, 127)
Val   : (11877, 127)


---
## 3. Entrainement XGBoost

In [6]:
xgb_params = {
    "objective": "reg:squarederror",
    "eta": 0.05,
    "min_child_weight": 360,
    "subsample": 0.85,
    "colsample_bytree": 0.3,
    "max_depth": 7,
    "verbosity": 0,
}
XGB_ROUNDS = 720

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)

print(f"Entrainement XGBoost ({XGB_ROUNDS} rounds)...")
xgb_model = xgb.train(
    xgb_params, dtrain, XGB_ROUNDS,
    evals=[(dtrain, "train"), (dval, "val")],
    verbose_eval=100,
)

xgb_val_preds = xgb_model.predict(dval)
xgb_train_preds = xgb_model.predict(dtrain)

# Score 1 : XGBoost arrondi simple
score_xgb_simple = qwk(xgb_val_preds, y_val)
print(f"\n[1] XGBoost arrondi simple  -> QWK = {score_xgb_simple:.4f}")

# Score 2 : XGBoost avec offsets
xgb_offsets = optimize_offsets(xgb_train_preds, y_train)
xgb_val_offset = apply_offsets(xgb_val_preds, xgb_offsets)
score_xgb_offset = qwk(xgb_val_offset, y_val)
print(f"[2] XGBoost avec offsets    -> QWK = {score_xgb_offset:.4f}  (gain : +{score_xgb_offset - score_xgb_simple:.4f})")

Entrainement XGBoost (720 rounds)...
[0]	train-rmse:2.42073	val-rmse:2.42197
[100]	train-rmse:1.85505	val-rmse:1.89683
[200]	train-rmse:1.81363	val-rmse:1.87018
[300]	train-rmse:1.79111	val-rmse:1.86317
[400]	train-rmse:1.77114	val-rmse:1.85958
[500]	train-rmse:1.75394	val-rmse:1.85883
[600]	train-rmse:1.73721	val-rmse:1.85805
[700]	train-rmse:1.72237	val-rmse:1.85866
[719]	train-rmse:1.71959	val-rmse:1.85856

[1] XGBoost arrondi simple  -> QWK = 0.5944
[2] XGBoost avec offsets    -> QWK = 0.6432  (gain : +0.0488)


---
## 4. Entrainement CatBoost

In [7]:
print("Entrainement CatBoost...")
cat_model = CatBoostRegressor(
    iterations=720,
    depth=7,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    task_type="CPU",
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))

cat_val_preds = cat_model.predict(X_val)
cat_train_preds = cat_model.predict(X_train)

# Score 3 : CatBoost arrondi simple
score_cat_simple = qwk(cat_val_preds, y_val)
print(f"\n[3] CatBoost arrondi simple -> QWK = {score_cat_simple:.4f}")

# Score 4 : CatBoost avec offsets
cat_offsets = optimize_offsets(cat_train_preds, y_train)
cat_val_offset = apply_offsets(cat_val_preds, cat_offsets)
score_cat_offset = qwk(cat_val_offset, y_val)
print(f"[4] CatBoost avec offsets   -> QWK = {score_cat_offset:.4f}  (gain : +{score_cat_offset - score_cat_simple:.4f})")

Entrainement CatBoost...
0:	learn: 2.4214123	test: 2.4225391	best: 2.4225391 (0)	total: 183ms	remaining: 2m 11s
100:	learn: 1.8747436	test: 1.9052387	best: 1.9052387 (100)	total: 1.75s	remaining: 10.7s
200:	learn: 1.8254124	test: 1.8688976	best: 1.8688976 (200)	total: 2.97s	remaining: 7.67s
300:	learn: 1.7922406	test: 1.8523969	best: 1.8523969 (300)	total: 4.18s	remaining: 5.81s
400:	learn: 1.7650694	test: 1.8453784	best: 1.8453585 (399)	total: 5.4s	remaining: 4.29s
500:	learn: 1.7421017	test: 1.8421514	best: 1.8421009 (499)	total: 6.58s	remaining: 2.88s
600:	learn: 1.7207445	test: 1.8394520	best: 1.8394095 (595)	total: 7.83s	remaining: 1.55s
700:	learn: 1.7012570	test: 1.8387156	best: 1.8384924 (682)	total: 9.19s	remaining: 249ms
719:	learn: 1.6972744	test: 1.8383469	best: 1.8383218 (716)	total: 9.45s	remaining: 0us

bestTest = 1.838321795
bestIteration = 716

Shrink model to first 717 iterations.

[3] CatBoost arrondi simple -> QWK = 0.6007
[4] CatBoost avec offsets   -> QWK = 0.6544

---
## 5. Comparaison des 4 configurations

In [8]:
results = {
    "[1] XGBoost simple":  score_xgb_simple,
    "[2] XGBoost offset":  score_xgb_offset,
    "[3] CatBoost simple": score_cat_simple,
    "[4] CatBoost offset": score_cat_offset,
}

print(f"{'=' * 50}")
print(f"RESULTATS (Quadratic Weighted Kappa sur validation)")
print(f"{'=' * 50}")
for name, score in results.items():
    print(f"  {name:<25} : {score:.4f}")
print(f"{'─' * 50}")

best_name = max(results, key=results.get)
best_score = results[best_name]
print(f"  Meilleur : {best_name} (QWK = {best_score:.4f})")

RESULTATS (Quadratic Weighted Kappa sur validation)
  [1] XGBoost simple        : 0.5944
  [2] XGBoost offset        : 0.6432
  [3] CatBoost simple       : 0.6007
  [4] CatBoost offset       : 0.6544
──────────────────────────────────────────────────
  Meilleur : [4] CatBoost offset (QWK = 0.6544)


---
## 6. Soumission (meilleur modele sur le test)

On re-entraine le meilleur modele sur TOUT le train, puis on predit sur le test.

In [9]:
use_offsets = "offset" in best_name
use_catboost = "CatBoost" in best_name

print(f"Re-entrainement sur tout le train ({X.shape[0]} lignes)...")
print(f"Modele : {'CatBoost' if use_catboost else 'XGBoost'} | Offsets : {'oui' if use_offsets else 'non'}")

if use_catboost:
    final_model = CatBoostRegressor(
        iterations=720, depth=7, learning_rate=0.05,
        loss_function="RMSE", random_seed=42, verbose=0, task_type="CPU",
    )
    final_model.fit(X, y)
    full_preds = final_model.predict(X)
    test_preds_raw = final_model.predict(X_test)
else:
    dfull = xgb.DMatrix(X, label=y)
    dtest = xgb.DMatrix(X_test)
    final_model = xgb.train(xgb_params, dfull, XGB_ROUNDS, verbose_eval=False)
    full_preds = final_model.predict(dfull)
    test_preds_raw = final_model.predict(dtest)

if use_offsets:
    final_offsets = optimize_offsets(full_preds, y)
    final_predictions = apply_offsets(test_preds_raw, final_offsets)
else:
    final_predictions = np.clip(np.round(test_preds_raw), 1, 8).astype(int)

print(f"\nPredictions generees : {len(final_predictions)}")
print(f"\nDistribution :")
print(pd.Series(final_predictions).value_counts().sort_index())

Re-entrainement sur tout le train (59381 lignes)...
Modele : CatBoost | Offsets : oui

Predictions generees : 19765

Distribution :
1    1508
2    1412
3    1107
4    2164
5    2019
6    1942
7    3406
8    6207
Name: count, dtype: int64


In [10]:
submission = pd.DataFrame({"Id": test_ids, "Response": final_predictions})
submission_path = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print(f"Fichier : {submission_path}")
print(f"Format  : {submission.shape}")
submission.head(10)

Fichier : prudential-life-insurance-assessment\submission.csv
Format  : (19765, 2)


,Id,Response
0,1,2
1,3,6
2,4,7
3,9,7
4,12,7
5,13,8
6,21,7
7,28,8
8,30,4
9,36,8


---
## Conclusion

| # | Configuration | QWK (validation) |
|:--|:-------------|:-----------------|
| 1 | XGBoost arrondi simple | voir ci-dessus |
| 2 | XGBoost + offsets | voir ci-dessus |
| 3 | CatBoost arrondi simple | voir ci-dessus |
| 4 | CatBoost + offsets | voir ci-dessus |

Le meilleur modele est utilise pour generer `submission.csv`.